In [10]:
import pandas as pd 
import os

In [7]:
data_path = 'C:/Users/dell/OneDrive/Desktop/inventory analytucs/'

product = pd.read_csv(data_path + "product.csv")
supplier = pd.read_csv(data_path + "supplier.csv")
customer = pd.read_csv(data_path + "customer.csv")
warehouse = pd.read_csv(data_path + "warehouse.csv")
carrier = pd.read_csv(data_path + "carrier.csv")

inventory = pd.read_csv(data_path + "inventory_snapshot.csv")
production = pd.read_csv(data_path + "production_run.csv")

purchase_order = pd.read_csv(data_path + "purchase_order.csv")
purchase_order_line = pd.read_csv(data_path + "purchase_order_line.csv")

shipment = pd.read_csv(data_path + "shipment.csv")
shipment_line = pd.read_csv(data_path + "shipment_line.csv")

In [8]:
datasets = {
    "product": product,
    "supplier": supplier,
    "customer": customer,
    "warehouse": warehouse,
    "carrier": carrier,
    "inventory": inventory,
    "production": production,
    "purchase_order": purchase_order,
    "purchase_order_line": purchase_order_line,
    "shipment": shipment,
    "shipment_line": shipment_line
}

for name, df in datasets.items():
    print(f"{name:25} {df.shape}")

product                   (40, 9)
supplier                  (20, 10)
customer                  (30, 8)
warehouse                 (4, 6)
carrier                   (4, 7)
inventory                 (16800, 8)
production                (4000, 9)
purchase_order            (3000, 7)
purchase_order_line       (7460, 7)
shipment                  (4500, 8)
shipment_line             (9059, 6)


In [9]:
for name, df in datasets.items():
    print("\n" + "=" * 50)
    print(name.upper())
    print("=" * 50)
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print("Missing values:")
    print(df.isnull().sum())


PRODUCT
Rows: 40
Columns: 9
Missing values:
product_id         0
product_code       0
category           0
unit_weight_kg     0
standard_cost      0
list_price         0
abc_class          0
hazardous          0
shelf_life_days    0
dtype: int64

SUPPLIER
Rows: 20
Columns: 10
Missing values:
supplier_id           0
supplier_name         0
country               0
tier                  0
lead_time_days        0
quality_rating        0
onboard_date          0
payment_terms_days    0
min_order_value       0
is_single_source      0
dtype: int64

CUSTOMER
Rows: 30
Columns: 8
Missing values:
customer_id         0
customer_name       0
industry_segment    0
region              0
priority_tier       0
credit_limit        0
contract_type       0
since_year          0
dtype: int64

WAREHOUSE
Rows: 4
Columns: 6
Missing values:
warehouse_id              0
warehouse_name            0
region                    0
capacity_units            0
operating_cost_monthly    0
automation_level          0
dtyp

In [11]:
for name, df in datasets.items():
    print(f"{name:25} duplicates = {df.duplicated().sum()}")

product                   duplicates = 0
supplier                  duplicates = 0
customer                  duplicates = 0
warehouse                 duplicates = 0
carrier                   duplicates = 0
inventory                 duplicates = 0
production                duplicates = 0
purchase_order            duplicates = 0
purchase_order_line       duplicates = 0
shipment                  duplicates = 0
shipment_line             duplicates = 0


In [12]:
print("Products:", product["product_id"].nunique(), len(product))
print("Suppliers:", supplier["supplier_id"].nunique(), len(supplier))
print("Customers:", customer["customer_id"].nunique(), len(customer))
print("Warehouses:", warehouse["warehouse_id"].nunique(), len(warehouse))
print("Carriers:", carrier["carrier_id"].nunique(), len(carrier))

Products: 40 40
Suppliers: 20 20
Customers: 30 30
Warehouses: 4 4
Carriers: 4 4


In [13]:
for name, df in {
    "inventory": inventory,
    "production": production,
    "purchase_order": purchase_order,
    "shipment": shipment
}.items():
    
    print("\n", name)
    
    for col in df.columns:
        if "date" in col:
            print(
                col,
                df[col].min(),
                "to",
                df[col].max()
            )


 inventory
snapshot_date 2024-01-01 to 2025-12-29

 production
run_date 2024-01-01 to 2025-12-30

 purchase_order
order_date 2024-01-01 to 2025-12-30
promised_date 2024-01-08 to 2026-02-03
actual_delivery_date 2024-01-06 to 2026-02-13

 shipment
ship_date 2024-01-01 to 2025-12-30
delivery_date 2024-01-06 to 2026-01-08


In [17]:
date_columns = {
    "supplier": ["onboard_date"],
    "purchase_order": [
        "order_date",
        "promised_date",
        "actual_delivery_date"
    ],
    "shipment": [
        "ship_date",
        "delivery_date"
    ],
    "production": ["run_date"],
    "inventory": ["snapshot_date"]
}

for table, columns in date_columns.items():
    for column in columns:
        datasets[table][column] = pd.to_datetime(
            datasets[table][column],
            errors="coerce"
        )

In [18]:
datasets["purchase_order"].dtypes

po_id                            int64
order_date              datetime64[us]
supplier_id                      int64
promised_date           datetime64[us]
actual_delivery_date    datetime64[us]
status                             str
expedited                        int64
dtype: object

In [19]:
#Delay in Delivery
po = datasets["purchase_order"]

po["delivery_delay_days"] = (
    po["actual_delivery_date"] -
    po["promised_date"]
).dt.days

In [20]:
#Visual of Delivery Date 
po[
    [
        "po_id",
        "order_date",
        "promised_date",
        "actual_delivery_date",
        "status",
        "delivery_delay_days"
    ]
].head(10)

,po_id,order_date,promised_date,actual_delivery_date,status,delivery_delay_days
0,1,2025-12-10,2025-12-17,2025-12-15,Delivered,-2.0
1,2,2024-03-10,2024-03-28,2024-03-28,Delivered,0.0
2,3,2025-04-01,2025-04-08,2025-04-18,Delivered,10.0
3,4,2025-02-16,2025-02-23,2025-02-23,Delivered,0.0
4,5,2024-01-11,2024-01-18,2024-01-18,Delivered,0.0
5,6,2025-07-03,2025-07-21,2025-07-24,Delivered,3.0
6,7,2025-02-09,2025-02-16,2025-02-14,Delivered,-2.0
7,8,2024-12-08,2024-12-26,2024-12-29,Delivered,3.0
8,9,2025-06-11,2025-06-29,2025-06-27,Delivered,-2.0
9,10,2025-04-10,2025-05-15,2025-05-17,Delivered,2.0


In [21]:
po["on_time_calculated"] = None

delivered = po["status"] == "Delivered"

po.loc[delivered, "on_time_calculated"] = (
    po.loc[delivered, "actual_delivery_date"]
    <=
    po.loc[delivered, "promised_date"]
).astype(int)

In [22]:
po_lines = datasets["purchase_order_line"]

In [25]:
#Total Ordered Value
po_lines["ordered_value"] = (
    po_lines["qty_ordered"] *
    po_lines["unit_cost"]
)

In [26]:
#total Recieved Value
po_lines["received_value"] = (
    po_lines["qty_received"] *
    po_lines["unit_cost"]
)

In [27]:
#Total Rejected Value
po_lines["rejected_value"] = (
    po_lines["qty_rejected"] *
    po_lines["unit_cost"]
)

In [28]:
#Recjection Rate
po_lines["rejection_rate"] = (
    po_lines["qty_rejected"] /
    po_lines["qty_ordered"]
) * 100

In [29]:
shipment = datasets["shipment"]
shipment_lines = datasets["shipment_line"]

In [30]:
shipment["transit_days"] = (
    shipment["delivery_date"] -
    shipment["ship_date"]
).dt.days

In [32]:
#Defect Rate
production = datasets["production"]

production["defect_rate"] = (
    production["defect_count"] /
    production["units_produced"]
) * 100

In [33]:
#Productivity
production["units_per_labor_hour"] = (
    production["units_produced"] /
    production["labor_hours"]
)

In [34]:
#Downtime
production["downtime_hours"] = (
    production["downtime_min"] / 60
)

In [36]:
inventory = datasets["inventory"]

In [37]:
#Inventory available
inventory["available_units"] = (
    inventory["units_on_hand"] -
    inventory["units_reserved"]
)

In [38]:
inventory["stock_status"] = "Healthy"

inventory.loc[
    inventory["available_units"] <= inventory["reorder_point"],
    "stock_status"
] = "Reorder"

inventory.loc[
    inventory["available_units"] == 0,
    "stock_status"
] = "Stockout"

In [39]:
import os

os.makedirs("data_cleaned", exist_ok=True)

In [40]:
for name, df in datasets.items():
    df.to_csv(
        f"data_cleaned/{name}_clean.csv",
        index=False
    )

In [41]:
import os

os.listdir("data_cleaned")

['carrier_clean.csv',
 'customer_clean.csv',
 'inventory_clean.csv',
 'production_clean.csv',
 'product_clean.csv',
 'purchase_order_clean.csv',
 'purchase_order_line_clean.csv',
 'shipment_clean.csv',
 'shipment_line_clean.csv',
 'supplier_clean.csv',
 'warehouse_clean.csv']

In [42]:
import os

os.listdir("data_cleaned")

['carrier_clean.csv',
 'customer_clean.csv',
 'inventory_clean.csv',
 'production_clean.csv',
 'product_clean.csv',
 'purchase_order_clean.csv',
 'purchase_order_line_clean.csv',
 'shipment_clean.csv',
 'shipment_line_clean.csv',
 'supplier_clean.csv',
 'warehouse_clean.csv']

In [43]:
for file in os.listdir("data_cleaned"):
    print(file)

carrier_clean.csv
customer_clean.csv
inventory_clean.csv
production_clean.csv
product_clean.csv
purchase_order_clean.csv
purchase_order_line_clean.csv
shipment_clean.csv
shipment_line_clean.csv
supplier_clean.csv
warehouse_clean.csv


In [44]:
pd.read_csv("data_cleaned/product_clean.csv").columns.tolist()

['product_id',
 'product_code',
 'category',
 'unit_weight_kg',
 'standard_cost',
 'list_price',
 'abc_class',
 'hazardous',
 'shelf_life_days']

In [45]:
for file in os.listdir("data_cleaned"):
    df = pd.read_csv("data_cleaned/" + file)
    print("\n", file)
    print(df.columns.tolist())


 carrier_clean.csv
['carrier_id', 'carrier_name', 'mode', 'cost_per_kg', 'avg_transit_days', 'co2_kg_per_km', 'reliability_score']

 customer_clean.csv
['customer_id', 'customer_name', 'industry_segment', 'region', 'priority_tier', 'credit_limit', 'contract_type', 'since_year']

 inventory_clean.csv
['snapshot_id', 'snapshot_date', 'warehouse_id', 'product_id', 'units_on_hand', 'reorder_point', 'units_reserved', 'units_in_transit', 'available_units', 'stock_status']

 production_clean.csv
['production_id', 'run_date', 'plant_id', 'product_id', 'units_produced', 'defect_count', 'downtime_min', 'labor_hours', 'shift', 'defect_rate', 'units_per_labor_hour', 'downtime_hours']

 product_clean.csv
['product_id', 'product_code', 'category', 'unit_weight_kg', 'standard_cost', 'list_price', 'abc_class', 'hazardous', 'shelf_life_days']

 purchase_order_clean.csv
['po_id', 'order_date', 'supplier_id', 'promised_date', 'actual_delivery_date', 'status', 'expedited', 'delivery_delay_days', 'on_time

In [46]:
import os

for file in os.listdir("data_cleaned"):
    df = pd.read_csv("data_cleaned/" + file)
    print("\n" + "="*60)
    print(file)
    print("="*60)
    print(df.columns.tolist())


carrier_clean.csv
['carrier_id', 'carrier_name', 'mode', 'cost_per_kg', 'avg_transit_days', 'co2_kg_per_km', 'reliability_score']

customer_clean.csv
['customer_id', 'customer_name', 'industry_segment', 'region', 'priority_tier', 'credit_limit', 'contract_type', 'since_year']

inventory_clean.csv
['snapshot_id', 'snapshot_date', 'warehouse_id', 'product_id', 'units_on_hand', 'reorder_point', 'units_reserved', 'units_in_transit', 'available_units', 'stock_status']

production_clean.csv
['production_id', 'run_date', 'plant_id', 'product_id', 'units_produced', 'defect_count', 'downtime_min', 'labor_hours', 'shift', 'defect_rate', 'units_per_labor_hour', 'downtime_hours']

product_clean.csv
['product_id', 'product_code', 'category', 'unit_weight_kg', 'standard_cost', 'list_price', 'abc_class', 'hazardous', 'shelf_life_days']

purchase_order_clean.csv
['po_id', 'order_date', 'supplier_id', 'promised_date', 'actual_delivery_date', 'status', 'expedited', 'delivery_delay_days', 'on_time_calcu

In [1]:
import os

os.listdir("data_cleaned")

['carrier_clean.csv',
 'customer_clean.csv',
 'inventory_clean.csv',
 'production_clean.csv',
 'product_clean.csv',
 'purchase_order_clean.csv',
 'purchase_order_line_clean.csv',
 'shipment_clean.csv',
 'shipment_line_clean.csv',
 'supplier_clean.csv',
 'warehouse_clean.csv']

In [2]:
import os

os.path.abspath("data_cleaned")

'C:\\Users\\dell\\data_cleaned'

In [10]:
import os
os.startfile(os.path.abspath("data_cleaned"))

In [4]:
import pandas as pd

df = pd.read_csv("data_cleaned/purchase_order_clean.csv")

print(df.columns.tolist())

['po_id', 'order_date', 'supplier_id', 'promised_date', 'actual_delivery_date', 'status', 'expedited', 'delivery_delay_days', 'on_time_calculated']


In [5]:
import pandas as pd

df = pd.read_csv("data_cleaned/purchase_order_clean.csv")

print(df.columns.tolist())
print(df.shape)

['po_id', 'order_date', 'supplier_id', 'promised_date', 'actual_delivery_date', 'status', 'expedited', 'delivery_delay_days', 'on_time_calculated']
(3000, 9)


In [6]:
print(df.head())

   po_id  order_date  supplier_id promised_date actual_delivery_date  \
0      1  2025-12-10            9    2025-12-17           2025-12-15   
1      2  2024-03-10           12    2024-03-28           2024-03-28   
2      3  2025-04-01           17    2025-04-08           2025-04-18   
3      4  2025-02-16            7    2025-02-23           2025-02-23   
4      5  2024-01-11            9    2024-01-18           2024-01-18   

      status  expedited  delivery_delay_days  on_time_calculated  
0  Delivered          0                 -2.0                 1.0  
1  Delivered          0                  0.0                 1.0  
2  Delivered          0                 10.0                 0.0  
3  Delivered          0                  0.0                 1.0  
4  Delivered          0                  0.0                 1.0  


In [7]:
import pandas as pd

df = pd.read_csv("data_cleaned/purchase_order_clean.csv")

df_sql = df[
    [
        "po_id",
        "order_date",
        "supplier_id",
        "promised_date",
        "actual_delivery_date",
        "status",
        "expedited"
    ]
]

df_sql.to_csv(
    "data_cleaned/purchase_order_sql.csv",
    index=False
)

print(df_sql.columns.tolist())
print(df_sql.shape)

['po_id', 'order_date', 'supplier_id', 'promised_date', 'actual_delivery_date', 'status', 'expedited']
(3000, 7)


In [9]:
import pandas as pd

# Load cleaned shipment data
df = pd.read_csv("data_cleaned/shipment_clean.csv")

# Keep only columns that belong in the MySQL shipment table
df_sql = df[
    [
        "shipment_id",
        "ship_date",
        "warehouse_id",
        "customer_id",
        "carrier_id",
        "delivery_date",
        "on_time_flag",
        "damaged_flag"
    ]
]

# Save SQL-ready file
df_sql.to_csv(
    "data_cleaned/shipment_sql.csv",
    index=False
)

# Verify
print("Columns:")
print(df_sql.columns.tolist())

print("\nShape:")
print(df_sql.shape)

print("\nFirst 5 rows:")
print(df_sql.head())

Columns:
['shipment_id', 'ship_date', 'warehouse_id', 'customer_id', 'carrier_id', 'delivery_date', 'on_time_flag', 'damaged_flag']

Shape:
(4500, 8)

First 5 rows:
   shipment_id   ship_date  warehouse_id  customer_id  carrier_id  \
0            1  2024-02-28             3           22           4   
1            2  2024-06-14             4            9           4   
2            3  2025-01-25             1           28           1   
3            4  2024-07-02             1           11           4   
4            5  2024-01-17             4           21           3   

  delivery_date  on_time_flag  damaged_flag  
0    2024-03-06             1             0  
1    2024-06-17             1             0  
2    2025-01-28             1             0  
3    2024-07-03             1             0  
4    2024-01-26             1             0  


In [11]:
import pandas as pd
import os

# --------------------------------------------------
# 1. INVENTORY
# --------------------------------------------------

inventory = pd.read_csv("data_cleaned/inventory_clean.csv")

inventory_sql = inventory.drop(
    columns=["available_units", "stock_status"],
    errors="ignore"
)

inventory_sql.to_csv(
    "data_cleaned/inventory_sql.csv",
    index=False
)


# --------------------------------------------------
# 2. PRODUCTION
# --------------------------------------------------

production = pd.read_csv("data_cleaned/production_clean.csv")

production_sql = production.drop(
    columns=[
        "defect_rate",
        "downtime_hours",
        "units_per_labor_hour"
    ],
    errors="ignore"
)

production_sql.to_csv(
    "data_cleaned/production_sql.csv",
    index=False
)


# --------------------------------------------------
# 3. PURCHASE ORDER LINE
# --------------------------------------------------

po_line = pd.read_csv(
    "data_cleaned/purchase_order_line_clean.csv"
)

po_line_sql = po_line.drop(
    columns=[
        "ordered_value",
        "received_value",
        "rejected_value",
        "rejection_rate"
    ],
    errors="ignore"
)

po_line_sql.to_csv(
    "data_cleaned/purchase_order_line_sql.csv",
    index=False
)


# --------------------------------------------------
# 4. VERIFY EVERYTHING
# --------------------------------------------------

files = [
    "inventory_sql.csv",
    "production_sql.csv",
    "purchase_order_line_sql.csv"
]

for file in files:
    df = pd.read_csv("data_cleaned/" + file)

    print("\n" + "=" * 60)
    print(file)
    print("=" * 60)
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print(df.columns.tolist())


inventory_sql.csv
Rows: 16800
Columns: 8
['snapshot_id', 'snapshot_date', 'warehouse_id', 'product_id', 'units_on_hand', 'reorder_point', 'units_reserved', 'units_in_transit']

production_sql.csv
Rows: 4000
Columns: 9
['production_id', 'run_date', 'plant_id', 'product_id', 'units_produced', 'defect_count', 'downtime_min', 'labor_hours', 'shift']

purchase_order_line_sql.csv
Rows: 7460
Columns: 7
['po_line_id', 'po_id', 'product_id', 'qty_ordered', 'qty_received', 'qty_rejected', 'unit_cost']
